# Seleção de participantes - Religião, pareamento estendido

Terceira etapa da seção 2.1 do paper: pareamento mais conservador entre religiosos e não religiosos, que acrescenta à distância medidas cognitivas e de personalidade.

**Regra de pareamento**

* Âncoras: `religious_now == 'yes'` (religiosos), percorridas na ordem da planilha.
* Para cada âncora, candidatos com `religious_now == 'no'` (não religiosos) e mesmos `sex`, `handedness`, `education_level`.
* Escolhe-se o candidato com menor média de `|Δ|` em `age`, `background_SES`, `IST_fluid`, `IST_intelligence_total`, `BIS`, `NEO_A` e `NEO_C` (coluna `dist`, valores ausentes ignorados), sem reposição.
* Resultado: 187 religiosos + 187 não religiosos = 374.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# Planilha com os 928 participantes do ID1000 (participants.tsv do OpenNeuro ds003097 + colunas usadas no projeto)
PARTICIPANTS_CSV = "https://docs.google.com/spreadsheets/d/13n8f5tILuIVF6Cxi2izX020_fYNjNNZk/export?format=csv"

# Pasta local de dados do projeto (a que contém ID1000Files/). Rode o notebook a partir dela ou ajuste aqui.
DATA_ROOT = "."
OUTPUT_DIR = DATA_ROOT

OUTPUT_XLSX = os.path.join(OUTPUT_DIR, "selected_participantsOtherVariables09-15.xlsx")

df = pd.read_csv(PARTICIPANTS_CSV)
df.shape

(928, 31)

Identificar sujeitos que não têm funcional

In [2]:
# Remove sujeitos que têm anat/ mas não têm func/ (sem fMRI funcional): 928 -> 881
base_dir = os.path.join(DATA_ROOT, "ID1000Files", "raw")

anat_only = []
for dataset in os.listdir(base_dir):
    dataset_dir = os.path.join(base_dir, dataset)
    if not os.path.isdir(dataset_dir):
        continue
    subs = [d for d in os.listdir(dataset_dir) if d.startswith("sub-") and os.path.isdir(os.path.join(dataset_dir, d))]
    for subj in subs:
        anat_dir = os.path.join(dataset_dir, subj, "anat")
        func_dir = os.path.join(dataset_dir, subj, "func")
        has_anat = os.path.isdir(anat_dir) and any(os.path.isfile(os.path.join(anat_dir, f)) for f in os.listdir(anat_dir))
        has_func = os.path.isdir(func_dir) and any(os.path.isfile(os.path.join(func_dir, f)) for f in os.listdir(func_dir))
        if has_anat and not has_func:
            anat_only.append(subj)

df = df[~df["participant_id"].isin(anat_only)]
print("sujeitos sem func:", len(anat_only), "| pool:", len(df))

sujeitos sem func: 47 | pool: 881


## Pareamento

In [3]:
# Pareamento 1:1 guloso, sem reposição
pool = df   # todos os 881 participantes com fMRI
VARS_DIST = ["age", "background_SES", "IST_fluid", "IST_intelligence_total", "BIS", "NEO_A", "NEO_C"]
ancoras = pool[pool["religious_now"] == "yes"].copy()      # religiosos
candidatos_pool = pool[pool["religious_now"] == "no"].copy()   # não religiosos

EXATAS = ['sex', 'handedness', 'education_level']   # variáveis pareadas exatamente

pares = []
for idx, row in ancoras.iterrows():
    candidatos = candidatos_pool
    for col in EXATAS:
        candidatos = candidatos[candidatos[col] == row[col]]
    if candidatos.empty:
        continue
    candidatos = candidatos.copy()

    # Distância: média das diferenças absolutas nas 7 variáveis (ignora valores ausentes)
    candidatos["dist"] = (candidatos[VARS_DIST] - row[VARS_DIST]).abs().mean(axis=1)
    mais_proximo = candidatos.sort_values("dist").iloc[0]

    pares.append(row)
    pares.append(mais_proximo)

    # Remove o já emparelhado
    candidatos_pool = candidatos_pool.drop(mais_proximo.name)

df_pares = pd.DataFrame(pares).reset_index(drop=True)
df_final = df_pares.sort_values(by=["age"])
print(df_final["religious_now"].value_counts())
df_final

religious_now
yes    187
no     187
Name: count, dtype: int64


,participant_id,age,sex,handedness,BMI,education_level,background_SES,IST_fluid,IST_memory,IST_crystallised,...,sexual_attraction_F,gender_identity_M,gender_identity_F,religious_upbringing,religious_now,religious_importance,DWI_TR_run1,DWI_TR_run2,DWI_TR_run3,dist
182,sub-0716,19.00,male,right,27,medium,3.0,82.0,41.0,13.0,...,7.0,7.0,1.0,no,yes,2.0,6.374,6.374,6.374,NaN
122,sub-0282,20.00,male,right,17,low,4.0,24.0,28.0,16.0,...,7.0,7.0,1.0,yes,yes,2.0,6.311,6.311,6.311,NaN
155,sub-0683,20.00,male,right,20,high,6.0,127.0,66.0,52.0,...,7.0,6.0,2.0,no,no,NaN,6.312,6.312,6.312,3.321429
98,sub-0056,20.00,male,right,21,medium,2.0,109.0,36.0,32.0,...,7.0,5.0,3.0,no,yes,2.0,6.311,6.311,NaN,NaN
190,sub-0772,20.00,female,left,22,medium,2.5,96.0,61.0,51.0,...,NaN,NaN,NaN,yes,yes,2.0,6.312,6.312,6.312,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,sub-0879,25.75,male,right,22,medium,2.0,125.0,50.0,40.0,...,7.0,6.0,2.0,no,no,NaN,6.311,6.311,6.311,2.178571
68,sub-0684,25.75,male,right,25,high,3.0,145.0,64.0,56.0,...,NaN,NaN,NaN,no,yes,1.0,6.313,6.313,6.313,NaN
4,sub-0061,25.75,female,right,18,medium,3.0,119.0,65.0,35.0,...,1.0,1.0,7.0,yes,yes,1.0,6.311,6.311,6.311,NaN
114,sub-0209,26.00,female,right,24,low,3.5,84.0,45.0,21.0,...,3.0,1.0,7.0,no,yes,2.0,6.311,6.311,6.311,NaN


## Verificação do balanço

In [4]:
# Balanço entre os grupos pareados
g1 = df_final[df_final["religious_now"] == "yes"]
g2 = df_final[df_final["religious_now"] == "no"]
for var in ["age", "background_SES", "IST_fluid", "IST_intelligence_total", "BIS", "NEO_A", "NEO_C", "NEO_O"]:
    t, p = stats.ttest_ind(g1[var].dropna(), g2[var].dropna(), equal_var=True)
    print(f"{var:24s} yes={g1[var].mean():.2f}  no={g2[var].mean():.2f}  t={t:.3f}  p={p:.3f}")
for var in EXATAS:
    print(var, "idêntico entre grupos:", (g1[var].value_counts().sort_index() == g2[var].value_counts().sort_index()).all())

age                      yes=22.74  no=22.90  t=-0.935  p=0.350
background_SES           yes=3.65  no=3.78  t=-1.120  p=0.263
IST_fluid                yes=99.74  no=100.05  t=-0.108  p=0.914
IST_intelligence_total   yes=189.54  no=189.38  t=0.035  p=0.972
BIS                      yes=19.69  no=19.41  t=0.738  p=0.461
NEO_A                    yes=41.60  no=40.81  t=1.430  p=0.154
NEO_C                    yes=39.21  no=38.52  t=1.192  p=0.234
NEO_O                    yes=39.39  no=40.50  t=-1.747  p=0.081
sex idêntico entre grupos: True
handedness idêntico entre grupos: True
education_level idêntico entre grupos: True


## Gravação

In [6]:
df_final.to_excel(OUTPUT_XLSX, index=False)